# Blind Screening Mitigation

This notebook tries a simple mitigation: remove the applicant's name, pronouns, and university from the resume text before scoring with SBERT. We then re-run the matching and compare against the original SBERT scores.

Important note about what this experiment can and cannot show: in our dataset, each counterfactual variant only differs from the original by the demographic field that was changed. If we delete that field from both versions, the two texts become identical, so the SBERT scores must be the same and the difference is zero by construction. This is an upper bound on how much demographic-only signal contributes to the score, not a realistic mitigation result for systems where demographic cues are mixed in with other text. We discuss this in the report.

In [ ]:
!pip install sentence-transformers pandas scikit-learn

In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [ ]:
from google.colab import files
uploaded = files.upload()
for filename in uploaded.keys():
    os.rename(filename, f"data/{filename}")
print("Files uploaded.")

In [ ]:
jobs = pd.read_csv("data/jobs.csv")
resumes = pd.read_csv("data/resume_variants.csv")
print("Jobs:", jobs.shape)
print("Resumes:", resumes.shape)

In [ ]:
# Build the blind version of each resume.
# We also strip the loose pronoun words (he/him/she/her/they/them) that appear
# inside sentences, since they can carry the same signal as the pronoun field.
PRONOUN_WORDS = ["she/her", "he/him", "they/them",
                 " she ", " he ", " they ",
                 " her ", " him ", " them ", " their "]

def make_blind_text(row):
    text = str(row["resume_text"])
    text = text.replace(str(row["name"]), "")
    text = text.replace(str(row["university"]), "")
    for w in PRONOUN_WORDS:
        text = text.replace(w, " ")
    return " ".join(text.split())

resumes["blind_resume_text"] = resumes.apply(make_blind_text, axis=1)
display(resumes[["resume_id", "version", "changed_signal", "blind_resume_text"]].head())

In [ ]:
def make_job_text(row):
    return f"{row['title']} {row['domain']} {row['company_name']} {row['job_description']}"

jobs["job_text"] = jobs.apply(make_job_text, axis=1)

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("SBERT loaded.")

In [ ]:
# Encode jobs once. Encode all blind resumes in one batch.
job_embeddings = model.encode(jobs["job_text"].tolist())
blind_resume_embeddings = model.encode(resumes["blind_resume_text"].tolist())
print("Embedding shapes:", blind_resume_embeddings.shape, job_embeddings.shape)

In [ ]:
# Pair every blind resume with every job.
rows = []
for i, resume_row in resumes.reset_index(drop=True).iterrows():
    for j, job_row in jobs.reset_index(drop=True).iterrows():
        score = float(cosine_similarity(
            blind_resume_embeddings[i].reshape(1, -1),
            job_embeddings[j].reshape(1, -1),
        )[0][0])
        rows.append({
            "resume_id": resume_row["resume_id"],
            "version": resume_row["version"],
            "changed_signal": resume_row["changed_signal"],
            "job_id": job_row["job_id"],
            "job_title": job_row["title"],
            "blind_similarity_score": score,
        })

blind_scores = pd.DataFrame(rows)
print("Blind scores:", len(blind_scores))

In [ ]:
# Compare each counterfactual blind score to its original blind score.
orig = (
    blind_scores[blind_scores["version"] == "original"]
    [["resume_id", "job_id", "job_title", "blind_similarity_score"]]
    .rename(columns={"blind_similarity_score": "blind_original_score"})
)
changed = blind_scores[blind_scores["version"] != "original"].rename(
    columns={"blind_similarity_score": "blind_changed_score"}
)

blind_comparison = changed.merge(orig, on=["resume_id", "job_id", "job_title"], how="left")
blind_comparison["blind_score_difference"] = (
    blind_comparison["blind_changed_score"] - blind_comparison["blind_original_score"]
)
blind_comparison["blind_absolute_difference"] = blind_comparison["blind_score_difference"].abs()
display(blind_comparison.head())

In [ ]:
blind_summary = blind_comparison.groupby("changed_signal").agg(
    average_blind_score_difference=("blind_score_difference", "mean"),
    average_blind_absolute_difference=("blind_absolute_difference", "mean"),
    max_blind_absolute_difference=("blind_absolute_difference", "max"),
    min_blind_score_difference=("blind_score_difference", "min"),
    max_blind_score_difference=("blind_score_difference", "max"),
).reset_index()
display(blind_summary)

## What these numbers actually mean

Because each counterfactual variant in our dataset differs from its original by only the single demographic field that was changed, removing that field from both versions makes the two texts identical and forces the score difference to zero. So a near-zero number here mostly confirms that SBERT is deterministic, not that blind screening would fix bias in a real ATS.

The pronoun result is the most useful one. Pronouns also appear inside normal sentences ("where she developed APIs"), and our blind step removes those too, but the residual difference if any tells us how much the surrounding text is still carrying gender signal.

In [ ]:
blind_scores.to_csv("results/blind_screening_scores.csv", index=False)
blind_comparison.to_csv("results/blind_screening_comparison.csv", index=False)
blind_summary.to_csv("results/blind_screening_summary.csv", index=False)
print("Saved.")

In [ ]:
from google.colab import files
files.download("results/blind_screening_scores.csv")
files.download("results/blind_screening_comparison.csv")
files.download("results/blind_screening_summary.csv")